# log_of_gravity

> 对应代码：`EconometricsCode/code_in_notes/Chap.4/log_of_gravity.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.4`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.4")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们使用贸易引力方程（gravity equation）数据集，把进出口额、双边 GDP、距离、共同边界、共同语言、殖民地关系、内陆国、 remoteness 与 WTO 相关变量等解释变量存入局部宏 explan_var，供各模型统一调用。

In [ ]:
%%stata
use ../datasets/Log_of_Gravity, clear
local explan_var "lypex lypim lyex lyim ldist border comlang colony landl_ex landl_im lremot_ex lremot_im comfrt_wto open_wto"

引力方程的传统做法是取对数后用 OLS 估计对数线性模型。由于贸易额存在大量零值，一种常见处理是用 log(1+trade)，但这种变换会改变系数的经济含义并引入偏误，我们先演示这种做法。

In [ ]:
%%stata
// log(1+y)
gen log_trade = log(1+trade)
reg log_trade `explan_var'

直接用 log(trade) 回归（数据中已存为 ltrade）则会把零贸易额的观测整行丢弃，造成样本选择问题——这正是在零值较多的贸易数据中不推荐对数线性化的原因。

In [ ]:
%%stata
// log(y)
reg ltrade `explan_var'

更合适的做法是保留水平值、直接对条件均值建模。nl 用非线性最小二乘估计 E(trade|x)=exp(xβ) 的指数形式，避免了取对数，但 NLS 在异方差下不一致，且没有利用计数数据的均值-方差关系。

In [ ]:
%%stata
// nonlinear least squares
nl (trade = exp({xb: `explan_var'} + {_cons}))

泊松伪极大似然（PPML）：以泊松似然估计 E(trade|x)=exp(xβ)。Santos Silva 和 Tenreyro 指出，只要条件均值设定正确，即使数据并非泊松分布，PPML 在异方差下仍然一致，且天然处理零贸易额，是引力方程的标准估计方法。

In [ ]:
%%stata
// ppml
poisson trade `explan_var'

负二项伪极大似然：nbreg 在泊松基础上引入过度离散参数，允许方差大于均值，作为对条件方差设定更灵活的替代；注意其一致性好于泊松的稳健性依赖更强的分布假定。

In [ ]:
%%stata
// negative binomial pml
nbreg  trade `explan_var'

最后我们用 Gamma 分布族加对数连接函数的广义线性模型（GLM）估计同样的指数均值函数。Gamma PML 适用于方差与均值平方成比例（常变异系数）的情形，与 PPML（方差正比于均值）对应不同的异方差结构，三者系数可以相互对照。

In [ ]:
%%stata
// gamma pml
glm trade `explan_var', family(gamma) link(log)